# GEMS G11 — Master Colab execution (real data)

One notebook runs the whole G11 multiresolution fault-discovery pipeline on the real GEMS data in Google Drive:
audit → 100 m geophysics → 1 m DEM → faults → features → EDA → geographic splits → XGBoost → SegFormer →
gated multiresolution fusion → model fusion → candidates → segment ranking → evaluation → official GeoTIFF → report.

* Every stage writes to `GEMS_2/outputs/G11_MULTISCALE_FAULT_DISCOVERY/` (Drive) with a manifest (config hash, source
  hashes, code commit, start / completion time, status, outputs + sha256, validation). Re-running a cell **reuses** a
  completed compatible stage and recomputes a stage whose inputs changed — a stage is never skipped because a file
  name exists.
* After a disconnect: run sections 00–03 again, then the section that was interrupted.
* `run_mode="SMOKE_TEST"` = LEVEL 2: a ~30 km real-data subset with faults AND DEM, 2 epochs, outputs in
  `..._SMOKE/`. Run it before `FULL` (LEVEL 3). LEVEL 1 = the synthetic test suite in section 03.
* DTI values here are the repository **offline DTI proxy** — not the official leaderboard DTI. Nothing is uploaded.

## 00 — Configuration

Edit only this cell. Unknown keys and incompatible values are rejected in section 03.

In [ ]:
CONFIG = {
    "repo": "mybenkhadda/GEMS---drivendata",
    "ref": "g11-colab-production",          # branch, tag or commit SHA that contains gems_bench/g11
    "drive_root": "/content/drive/MyDrive/GEMS_2",
    "repo_root": "/content/GEMS",
    "local_root": "/content/g11_work",

    "run_mode": "SMOKE_TEST",               # AUDIT_ONLY | INGEST_ONLY | EDA_ONLY | TRAIN_ONLY | EVALUATE_ONLY |
                                            # SUBMISSION_ONLY | FULL | SMOKE_TEST  (run SMOKE_TEST before FULL)
    "resume": True,
    "force_recompute": False,

    "use_gpu": True,
    "mixed_precision": True,
    "seed": 2026,

    "run_audit": True, "run_ingestion": True, "run_features": True, "run_eda": True, "run_splits": True,
    "run_training": True, "run_fusion": True, "run_candidates": True, "run_evaluation": True, "run_submission": True,

    "models": ["xgboost", "segformer", "gated_fusion", "segment_ranker"],

    "dem_native_resolution_m": 1.0,
    "dem_statistics_resolution_m": 2.0,
    "dem_training_resolution_m": 25.0,
    "geo_resolution_m": 100.0,

    "training_target_buffer_m": 200.0,
    "unlabeled_background_weight": 0.3,

    "strict_integrity": True,
    "failure_policy": "STOP_REQUIRED_SKIP_OPTIONAL",

    "save_intermediate": True,
    "generate_figures": True,
    "generate_html_report": True,

    "official_submission": True,
    "upload_submission": False,              # must stay False: G11 never uploads
    "smoke_window_km": 30.0,
}

## 01 — Drive mount + runtime detection

In [ ]:
import os, sys, time, json, shutil, subprocess, re
from pathlib import Path
T_START = time.time()
from google.colab import drive
drive.mount("/content/drive")
if not Path(CONFIG["drive_root"]).is_dir():
    raise RuntimeError(f"Drive root {CONFIG['drive_root']} not found: check the path or remount "
                       "(drive.mount('/content/drive', force_remount=True))")
Path(CONFIG["local_root"]).mkdir(parents=True, exist_ok=True)
def sh(*args, check=True, cwd=None):
    r = subprocess.run(list(args), capture_output=True, text=True, cwd=cwd)
    if check and r.returncode:
        raise RuntimeError(f"command failed: {' '.join(args)}\n{r.stderr[-2000:]}")
    return r.stdout.strip()
print(sh("nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader", check=False) or "no GPU")
print("RAM:", sh("free", "-g", check=False).splitlines()[1] if shutil.which("free") else "?")
print("local disk free (GB):", round(shutil.disk_usage("/content").free / 1e9, 1))
print("drive mounted:", Path(CONFIG["drive_root"]).is_dir())

## 02 — Repository checkout + dependencies

Clones `repo_root` if absent and checks out `ref` **detached**. If the checkout has local modifications it is
**not** overwritten (the cell stops). If `ref` does not exist on GitHub, or lacks the G11 package, the cell stops with
the deployment commands to run on the machine that holds the code.

In [ ]:
REPO_URL = f"https://github.com/{CONFIG['repo']}.git"
REF, ROOT = CONFIG["ref"], Path(CONFIG["repo_root"])
DEPLOY_HELP = f"""
The G11 code is not available at {REPO_URL} @ {REF}.
On the machine with the working copy (the G11 files were uncommitted when this notebook was written):
    git checkout -b {REF}
    git add gems_bench/g11 configs/g11_multiscale.yaml docs/g11 notebooks/colab/GEMS_G11_*.ipynb
    git commit -m "G11 multiscale fault discovery + Colab master execution"
    git push -u origin {REF}
then re-run this cell (or set CONFIG['ref'] to a branch / commit that contains gems_bench/g11).
"""
is_sha = re.fullmatch(r"[0-9a-f]{7,40}", REF) is not None
remote = sh("git", "ls-remote", "--heads", "--tags", REPO_URL, REF, check=False)
if not remote and not is_sha:
    raise RuntimeError(DEPLOY_HELP)
if not (ROOT / ".git").exists():
    sh("git", "clone", "--filter=blob:none", REPO_URL, str(ROOT))
dirty = sh("git", "status", "--porcelain", "--untracked-files=no", cwd=str(ROOT))
if dirty:
    raise RuntimeError(f"{ROOT} has local modifications - not overwriting them:\n{dirty}\n"
                       "commit / stash them, or set CONFIG['repo_root'] to a fresh path")
sh("git", "fetch", "--depth", "1", "origin", REF, cwd=str(ROOT))
sh("git", "checkout", "--detach", "FETCH_HEAD", cwd=str(ROOT))
COMMIT = sh("git", "rev-parse", "HEAD", cwd=str(ROOT))
REQUIRED = ["gems_bench/g11/pipeline.py", "gems_bench/g11/notebook_config.py", "gems_bench/g11/colab_runtime.py",
            "gems_bench/g11/external_predictions.py", "configs/g11_multiscale.yaml"]
missing = [f for f in REQUIRED if not (ROOT / f).is_file()]
if missing:
    raise RuntimeError(f"{REF} ({COMMIT[:10]}) lacks {missing}." + DEPLOY_HELP)
sh(sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[colab]", "xgboost", "lightgbm", "psutil", "pyarrow")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import gems_bench.g11 as G11
from gems_bench.g11 import pipeline as G11P, notebook_config as NC, colab_runtime as CR
print("G11", G11.__version__, "| commit", COMMIT)

## 03 — Tests + environment integrity (LEVEL 1)

The synthetic integrity suite must pass before anything touches real data. Then the notebook CONFIG is validated and adapted to the YAML, and the runtime is configured (only batch size / AMP / workers / chunk sizes change; every adjustment is recorded).

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", str(ROOT / "gems_bench/g11/tests"), "-q", "-p", "no:cacheprovider",
                    "-x", "--ignore", str(ROOT / "gems_bench/g11/tests/test_real_data.py")],
                   capture_output=True, text=True, cwd=str(ROOT))
print(r.stdout[-2500:])
if r.returncode != 0:
    raise RuntimeError("LEVEL 1 synthetic tests failed - stop (see output above)")

from gems_bench.g11.config import load_config
BASE = load_config()
PLAN = NC.build_plan(CONFIG, BASE)
CFG = load_config(overrides=PLAN.overrides)
INFO = CR.detect_runtime(CONFIG["drive_root"], CONFIG["local_root"])
CFG, ADJUSTMENTS = CR.configure_runtime(CFG, INFO, CONFIG["use_gpu"], CONFIG["mixed_precision"])
P = G11P.G11Pipeline(CFG)
RUN = NC.StepRunner(P, PLAN)
CR.save_runtime_report(P.ctx.d("runtime") / "runtime_report.json", INFO, ADJUSTMENTS, P.tracker)
print(json.dumps({k: INFO.get(k) for k in ("gpu_name", "gpu_memory_gb", "cuda_version", "ram_total_gb",
                                             "local_disk_free_gb", "drive_mounted")}, indent=1, default=str))
print("adjustments:", ADJUSTMENTS)
print("notes:", PLAN.notes)
print("output root:", P.ctx.root, "| config hash", P.ctx.cfg_hash)
display(PLAN.table())

## 04 — Artifact discovery

Read-only discovery of required and optional sources (nothing is assumed from a file name; arrays need documented placement).

In [ ]:
from gems_bench.g11 import audit as A
REG, SUMMARY = A.build_artifact_registry(CFG)
print(json.dumps(SUMMARY["found"], indent=1))
print("optional evidence:", SUMMARY["optional"])
print("skipped:", *SUMMARY["skipped"], sep="\n  ")
if SUMMARY["problems"]:
    print("PROBLEMS (the audit will stop):", *SUMMARY["problems"], sep="\n  ")

## 05 — Real-data audit

Fail-fast checks (CRS, grid, template), DEM vertical units from authoritative metadata, existing predictions (G07 arrays placed via their adjacent `manifest.json`, nested-grid check, provenance), SMOKE window.

In [ ]:
AUD = RUN("audit", P.audit, show={"data_root": CFG["data_root"], "smoke": CFG["smoke"]},
          check=lambda s: None if s["grid"]["epsg"] == 32611 else "master grid is not EPSG:32611")
import pandas as pd
from IPython.display import Markdown
display(Markdown((P.ctx.d("audit") / "audit_report.md").read_text()))
display(pd.read_csv(P.ctx.d("audit") / "dem_vertical_units.csv")["class"].value_counts())
try:
    display(pd.read_csv(P.ctx.d("audit") / "artifact_compatibility.csv")[["family", "name", "status", "provenance"]])
except Exception as e:
    print("no external predictions:", e)

## 06 — 100 m geophysics ingestion

Master grid, band registry (all GeoDAWN bands, names, units, finite fractions), fault registry with fault-system grouping, DEM catalog, coverage report and visual checks.

In [ ]:
ING = RUN("ingest", P.ingest, show={"expected_band_count": CFG["geophysical_grid"]["expected_band_count"]},
          check=lambda s: None if s["fault_pixels"] > 0 else "no mapped fault pixel in the grid")
display(pd.read_parquet(P.ctx.d("ingest") / "geophysical_registry.parquet"))
from IPython.display import Image
for f in sorted((P.ctx.d("ingest") / "figures").glob("*.png"))[:4]:
    display(Image(str(f)))

## 07 — 1 m DEM catalog + coverage

In [ ]:
cov = json.loads((P.ctx.d("ingest") / "coverage_report.json").read_text())
print(json.dumps(cov, indent=1))
cat_p = P.ctx.d("ingest") / "dem_catalog.parquet"
if cat_p.is_file():
    cat = pd.read_parquet(cat_p)
    display(cat[["tile_id", "epsg", "resolution_m", "year", "vertical_unit", "quality_flags", "priority_rank"]].head(20))
for f in sorted((P.ctx.d("ingest") / "figures").glob("dem_*.png")) + sorted((P.ctx.d("ingest") / "figures").glob("native_*.png")):
    display(Image(str(f)))

## 08 — Fault integration

In [ ]:
fr = pd.read_parquet(P.ctx.d("ingest") / "fault_registry.parquet")
print(len(fr), "segments;", fr.get("fault_system_id", pd.Series(dtype=int)).nunique(), "fault systems")
display(fr.head(10))
for f in sorted((P.ctx.d("ingest") / "figures").glob("band_*_faults.png"))[:1]:
    display(Image(str(f)))

## 09 — Multiresolution feature engineering

Streaming 100 m geophysics (memory-bounded), exact 1 m → 100 m DEM aggregation (2 m work resolution, halo-aware chunks), multimodal alignment, cross-fault morphology of known faults and native terrain windows. Label-derived layers are registered and can never be predictors.

In [ ]:
print(CR.estimate_stage_memory("features", {"grid_shape": P.grid.shape}))
print(CR.estimate_stage_memory("dem_aggregate", {"chunk_cells": CFG["dem"]["aggregate_chunk_cells"], "res_m": CFG["dem"]["stats_resolution_m"]}))
FEAT = RUN("features", P.features,
           check=lambda s: None if set(P.store.predictors()).isdisjoint(
               P.store.reg[P.store.reg.label_derived.astype(bool)].name) else "label-derived layer among predictors")
display(P.store.reg.groupby("family").agg(n=("name", "size"), finite=("finite_fraction", "mean")))

## 10 — Geological EDA

Coverage, distributions, multiscale contrast (100 m and native DEM), existing-model evidence (provenance-labelled), failure taxonomy, ≥ 20-site candidate atlas, evidence-based modeling recommendations.

In [ ]:
EDA = RUN("eda", P.eda)
d = P.ctx.d("eda")
if (d / "eda_summary.md").is_file():
    display(Markdown((d / "eda_summary.md").read_text()))
    for f in ["fig_coverage.png", "fig_contrast.png", "fig_utility.png"]:
        if (d / f).is_file(): display(Image(str(d / f)))
    print("atlas sites:", len(list((d / "candidate_atlas").glob("site_*.png"))), "| HTML:", d / "eda_report.html")

## 11 — Geographic splits

Train / validation / stack (calibration) / test blocks with fault-system grouping and embargo. **Training stops if any leakage check fails.**

In [ ]:
SPL = RUN("splits", P.splits, check=lambda s: None if json.loads((P.ctx.d("splits") / "leakage_report.json").read_text())["passed"]
          else "leakage checks failed")
display(Image(str(P.ctx.d("splits") / "split_map.png")))
print(json.dumps(json.loads((P.ctx.d("splits") / "leakage_report.json").read_text()), indent=1)[:2000])

## 12 — XGBoost baselines (E0 raw, E1 geophysics, E2 + 100 m DEM summaries)

PU-weighted unlabeled background, early stopping on validation, identical splits for all three.

In [ ]:
X = [e for e in PLAN.experiments if e in ("E0", "E1", "E2")]
print(CR.estimate_stage_memory("train_pixel", {"n_samples": CFG["sampling"]["pixel_samples_per_split"],
                                               "n_features": len(P.store.predictors()), "grid_shape": P.grid.shape}))
TXGB = RUN("train_pixel", lambda: P.train(X), show={"experiments": X, "target": CFG["labels"]["training_target"]})
display(TXGB)

## 13 — SegFormer training (E3 + loss ablation)

Mixed precision, checkpoint per epoch (`06_models/segformer/`), early stopping and selection on validation; inputs staged to local scratch.

In [ ]:
print(CR.estimate_stage_memory("train_deep", {"batch_size": CFG["training"]["batch_size"], "patch_px": CFG["sampling"]["patch_px"],
                                              "channels": CFG["models"]["segformer"].get("in_channels") or 16, "grid_shape": P.grid.shape}))
TSEG = RUN("train_segformer", lambda: P.train(["E3"]), show={"loss": CFG["training"]["loss"],
           "ablation": CFG["training"].get("loss_ablation"), "batch": CFG["training"]["batch_size"], "amp": CFG["training"]["amp"]})
display(TSEG)

## 14 — Gated multiresolution fusion (E4 + mode ablation)

100 m geophysics + DEM patches at `dem_training_resolution_m` with a DEM-availability mask; compares geophysics-only, DEM-only, concatenation and gated.

In [ ]:
TGF = RUN("train_gated", lambda: P.train(["E4"]), show={"mode": CFG["models"]["gated_fusion"]["mode"],
          "ablation": CFG["models"]["gated_fusion"].get("mode_ablation"), "dem_res_m": CFG["models"]["terrain_encoder"]["dem_res_m"]})
display(TGF)

## 15 — Candidate generation (after model fusion + validation selection)

Order of operations (the candidates need to know which map to use, chosen on VALIDATION only):
1. pixel-level model fusion of the G11 maps (fitted on stack, selected on validation → `E5`);
2. validation leaderboard + `selection.json` (written before any test metric);
3. candidates from the selected calibrated map AND from label-free geophysical lineaments → linking → deduplication.

In [ ]:
FUS = RUN("fusion", P.fusion)
if FUS is not None: display(FUS)
LB = RUN("evaluate", lambda: P.evaluate(final=False))
CAND = RUN("candidates", P.infer, check=lambda m: None if m["n_candidates"] > 0 else "no candidate")
print(CAND)

## 16 — Segment ranking

LightGBM binary vs lambdarank (chosen on validation), length budgets compared on the validation offline DTI proxy.

In [ ]:
d = P.ctx.d("infer")
if (d / "ranker_validation.parquet").is_file():
    display(pd.read_parquet(d / "ranker_validation.parquet"))
    display(pd.read_parquet(d / "budget_validation.parquet"))

## 17 — Model comparison (validation leaderboard)

All experiments on identical support and the same label definition; spatial block-bootstrap CI on PR-AUC.

In [ ]:
if LB is not None:
    cols = ["experiment_id", "model", "PR_AUC", "PR_AUC_ci_low", "PR_AUC_ci_high", "ROC_AUC", "calibration_error",
            "candidate_recall", "predicted_length_km", "fragmentation", "offline_DTI_proxy", "training_time_s", "peak_GPU_memory_gb"]
    display(LB[[c for c in cols if c in LB]].sort_values("PR_AUC", ascending=False))
    print(json.loads((P.ctx.d("evaluate") / "selection.json").read_text()))

## 18 — Offline DTI proxy evaluation (held-out test, once)

Refused unless the validation selection exists. The offline DTI proxy is **not** the official DTI.

In [ ]:
FIN = RUN("evaluate", lambda: P.evaluate(final=True))
if FIN is not None:
    display(FIN[FIN.split == "test"][["experiment_id", "PR_AUC", "ROC_AUC", "candidate_recall", "offline_DTI_proxy"]])

## 19 — Official GeoTIFF submission

Single-band float32 on the official template grid, values in [0, 1], written with the repository V1 writer, reopened and validated, SHA-256 + manifest. `official_DTI = unavailable` unless an official evaluator is installed. **No upload.** In SMOKE_TEST the file is a format check only (NOT_FOR_UPLOAD).

In [ ]:
SUB = RUN("submit", P.submit, check=lambda r: None if r["validation"]["passed"] else "GeoTIFF validation failed")
if SUB is not None:
    print(json.dumps({k: SUB[k] for k in ("path", "sha256", "validation", "official_DTI", "uploaded")}, indent=1, default=str))

## 20 — Final report + Drive synchronization

In [ ]:
REP = RUN("report", lambda: P.report(PLAN, INFO))
if REP is not None:
    display(Markdown(Path(REP["paths"]["md"]).read_text()[:20000]))
    print("HTML:", REP["paths"]["html"], "| leaderboard:", REP["paths"]["leaderboard"])
os.sync()                                         # flush pending writes to the Drive FUSE mount
print("step results:", json.dumps(RUN.results, indent=1))
print(f"total notebook time: {(time.time() - T_START) / 3600:.2f} h | outputs: {P.ctx.root}")